# Refinement results and final submission checks

Read-only review of v004/v004b. This notebook does not fit or submit a model.
Lower inspiratory MAE is better. Split A selected models, epochs and ensemble
weights; split B is a grouped resampling confirmation/filtering stage. Final
all-labeled models have **no independent A/B holdout**. Their earlier validation
models provide the reported scores.

The original v004 TCN was stopped because of an inefficient GPU fallback. v004b
uses an equivalent, verified convolution implementation and imports four completed
non-TCN trials with provenance checks. Imported results are counted once.


In [ ]:
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
PROJECT=Path.cwd().resolve()
if not (PROJECT/'data').is_dir():PROJECT=PROJECT/'ventilator-pressure-prediction'
OUT=PROJECT/'outputs/sequence_v004b';EXP=OUT/'experiments';FINAL=OUT/'final_all'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def read(path):return json.loads(Path(path).read_text())
records=[read(p) for p in EXP.glob('record_*.json')]
print('Completed validation model records:',len(records))
results=pd.DataFrame([{k:r.get(k) for k in ['name','split','validation_mae','best_epoch','training_breaths','elapsed_seconds','reused_from_parent']} for r in records])
if not results.empty:display(results.sort_values(['split','validation_mae']).reset_index(drop=True))


## Learning behavior

These curves show native-unit inspiratory MAE. Training MAE is measured online
while weights change; validation uses the fixed epoch-end model. Their difference
alone does not establish overfitting. The selected checkpoint uses the lowest
validation MAE, including epoch zero for warm starts. A time-limited TCN comparison
does not prove which architecture would win with a larger training budget.


In [ ]:
fig,axes=plt.subplots(2,3,figsize=(13,7),constrained_layout=True)
for ax,name in zip(axes.flat,['warm_current','warm_tenth','bigru80','gru_embedding','tcn_lr15','tcn_lr30']):
    path=EXP/('A_'+name+'_epochs.jsonl')
    if not path.exists():path=PROJECT/'outputs/sequence_v004/experiments'/path.name
    if not path.exists():ax.set_title(name+' — not started');continue
    events=[json.loads(s) for s in path.read_text().splitlines() if s.strip()]
    epochs=[e for e in events if e.get('event')=='epoch']
    if epochs:
        x=[e['epoch'] for e in epochs]
        ax.plot(x,[e['training_online_inspiratory_mae'] for e in epochs],label='Training, online',alpha=.8)
        ax.plot(x,[e['validation_inspiratory_mae'] for e in epochs],label='Validation A')
        best=min(epochs,key=lambda e:e['validation_inspiratory_mae'])
        ax.scatter(best['epoch'],best['validation_inspiratory_mae'],s=22,color='black',zorder=4)
    ax.set_title(name);ax.set_xlabel('Epoch / extra epoch');ax.set_ylabel('Inspiratory MAE');ax.grid(alpha=.2)
axes.flat[0].legend(fontsize=8)
plt.show()


## Frozen ensemble and second-split confirmation

Weights are selected on A only and frozen before fresh B fits. New individual
models must improve the reference on both A and B. An ensemble must improve
the best individual on both splits. B is not used to optimize weights or stop
training, but it is used by this stated eligibility rule.


In [ ]:
if (EXP/'locked_before_B.json').exists():display(read(EXP/'locked_before_B.json'))
else:print('A model comparison is still running; no ensemble is locked yet.')
if (EXP/'approach_comparison.csv').exists():display(pd.read_csv(EXP/'approach_comparison.csv'))
if (EXP/'validation_selection.json').exists():display(read(EXP/'validation_selection.json'))
else:print('Second-split confirmation is not complete.')


## Independent artifact checks

Recompute every saved validation MAE against the approved labels by exact row ID;
verify both grouped splits, prediction and checkpoint hashes, and final CSV
schema/order/finiteness. All-labeled final fits are verified as a separate stage.


In [ ]:
checks={}
if records:
    train=pd.read_csv(PROJECT/'data/train.csv',usecols=['id','breath_id','pressure','u_out'])
    ids=train.id.to_numpy();assert train.id.is_unique
    groups=np.load(EXP/'split_breath_ids.npz')
    train_breaths=train.breath_id.to_numpy().reshape(-1,80)[:,0]
    assert len(train_breaths)==67905 and np.array_equal(groups['final_all'],train_breaths)
    assert not np.intersect1d(groups['A_fit'],groups['A_validation']).size
    assert not np.intersect1d(groups['B_fit'],groups['A_validation']).size
    assert not np.intersect1d(groups['B_fit'],groups['B_validation']).size
    for split in ['A','B']:
        if not (EXP/(split+'_row_ids.npy')).exists():continue
        rows=np.load(EXP/(split+'_row_ids.npy'))
        loc=pd.Index(ids).get_indexer(rows);assert (loc>=0).all()
        truth=train.iloc[loc];assert (truth.u_out==0).all()
        assert set(truth.breath_id)==set(groups[split+'_validation'])
        assert len(groups[split+'_validation'])==6791
        for r in [r for r in records if r['split']==split]:
            pred=np.load(r['prediction_file']);assert pred.shape==rows.shape and np.isfinite(pred).all()
            assert hashlib.sha256(pred.tobytes()).hexdigest()==r['prediction_sha256']
            metric=float(np.abs(pred-truth.pressure.to_numpy(dtype=np.float32)).mean())
            assert np.isclose(metric,r['validation_mae'],atol=1e-9,rtol=0),r['name']
            assert sha(r['checkpoint_path'])==r['checkpoint_sha256']
    checks.update(validation_scores_recomputed=len(records),grouped_splits='passed',prediction_and_checkpoint_hashes='passed')
if (FINAL/'final_manifest.json').exists():
    manifest=read(FINAL/'final_manifest.json');selection=read(EXP/'validation_selection.json')
    assert manifest['training_breaths']==67905 and not manifest['final_refit_has_independent_holdout']
    assert manifest['selected_members']==selection['members'] and manifest['weights']==selection['weights']
    assert manifest['validation_evidence_sha256']==sha(EXP/'validation_selection.json')
    sample=pd.read_csv(PROJECT/'data/sample_submission.csv');test_ids=pd.read_csv(PROJECT/'data/test.csv',usecols=['id'])
    csv=pd.read_csv(FINAL/'submission.csv')
    assert csv.columns.tolist()==['id','pressure'] and len(csv)==603600
    assert csv.id.is_unique and np.array_equal(csv.id,sample.id) and set(csv.id)==set(test_ids.id)
    assert np.isfinite(csv.pressure).all() and sha(FINAL/'submission.csv')==manifest['submission_sha256']
    for model in manifest['selected_members']:
        r=read(FINAL/('record_'+model+'.json'));a=read(EXP/('record_A_'+model+'.json'))
        assert r['training_breaths']==67905 and not r['validation_used'] and not r['has_independent_holdout']
        assert sha(r['checkpoint_path'])==r['checkpoint_sha256']
        if r['kind']!='reference_tree':assert r['report']['executed_epochs']==len(a['selected_lr_schedule'])
    checks.update(final_fit_breaths=67905,final_csv_rows=len(csv),final_csv_sha256=manifest['submission_sha256'],final_csv_checks='passed')
    print('Final all-labeled CSV:',FINAL/'submission.csv')
else:print('Final all-labeled fitting / CSV checks are pending.')
display(checks)
